In [ ]:
import os
import json
import random
import warnings
from typing import List, Dict, Tuple, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset

warnings.filterwarnings("ignore", message=".*VisibleDeprecationWarning.*")
warnings.filterwarnings("ignore", message=".*dtype\\(\\): align should be passed.*")

QUICK_RUN = False
SAVE_PATH = "./outputs/baseline_replay_10task.json"
REPLAY_BUFFER_CAPACITY = 2000
N_TASKS = 10
CLASSES_PER_TASK = 10
TOTAL_CLASSES = N_TASKS * CLASSES_PER_TASK
EPOCHS_BASE = 200 if not QUICK_RUN else 5
EPOCHS_INCREMENTAL = 80 if not QUICK_RUN else 5
BATCH_SIZE = 128
USE_AMP = True
FORCE_CPU = False
NUM_WORKERS = 2


def set_deterministic(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def get_device(force_cpu: bool = False) -> torch.device:
    if force_cpu or not torch.cuda.is_available():
        return torch.device("cpu")
    return torch.device("cuda")


def save_progress(obj: Dict, path: str = SAVE_PATH) -> None:
    directory = os.path.dirname(path)
    if directory:
        os.makedirs(directory, exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)


class CIFAR100TaskDataset(Dataset):
    def __init__(self, dataset: Dataset, classes: List[int]):
        self.dataset = dataset
        self.classes = classes
        self.indices = [i for i, label in enumerate(dataset.targets) if label in classes]

    def __len__(self) -> int:
        return len(self.indices)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, int]:
        img, label = self.dataset[self.indices[idx]]
        return img, label


class TensorDatasetWrapper(Dataset):
    def __init__(self, images: torch.Tensor, labels: torch.Tensor):
        self.images = images
        self.labels = labels

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, int]:
        return self.images[idx], self.labels[idx]


def get_task_classes(task_id: int) -> List[int]:
    start_class = (task_id - 1) * CLASSES_PER_TASK
    end_class = task_id * CLASSES_PER_TASK
    return list(range(start_class, end_class))


def get_task_loaders(
    train_ds: Dataset,
    test_ds: Dataset,
    task_id: int,
    batch_size: int = BATCH_SIZE,
    num_workers: int = NUM_WORKERS,
) -> Tuple[DataLoader, DataLoader]:
    task_classes = get_task_classes(task_id)
    train_subset = CIFAR100TaskDataset(train_ds, task_classes)
    test_subset = CIFAR100TaskDataset(test_ds, task_classes)
    pin_memory = torch.cuda.is_available() and not FORCE_CPU

    train_loader = DataLoader(
        train_subset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=pin_memory,
    )
    test_loader = DataLoader(
        test_subset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin_memory,
    )
    return train_loader, test_loader


class DynamicReplayBuffer:
    def __init__(self, capacity_per_class: int = REPLAY_BUFFER_CAPACITY):
        self.capacity_per_class = capacity_per_class
        self.storage: Dict[int, List[Tuple[torch.Tensor, float]]] = {}

    def add_exemplars_from_dataset(
        self,
        dataset: Dataset,
        classes: List[int],
        feature_extractor: nn.Module,
        device: torch.device,
        batch_size: int = BATCH_SIZE,
        num_workers: int = NUM_WORKERS,
    ) -> None:
        pin_memory = torch.cuda.is_available() and not FORCE_CPU
        loader = DataLoader(
            CIFAR100TaskDataset(dataset, classes),
            batch_size=batch_size,
            shuffle=False,
            num_workers=num_workers,
            pin_memory=pin_memory,
        )

        by_class_images: Dict[int, List[torch.Tensor]] = {c: [] for c in classes}
        by_class_feats: Dict[int, List[torch.Tensor]] = {c: [] for c in classes}

        feature_extractor.eval()
        with torch.no_grad():
            for x, y in loader:
                x = x.to(device)
                feats = feature_extractor(x)
                feats = F.normalize(F.adaptive_avg_pool2d(feats, 1).flatten(1), p=2, dim=1).cpu()
                for img, label, feat in zip(x.cpu(), y.cpu(), feats):
                    label_i = int(label.item())
                    by_class_images[label_i].append(img)
                    by_class_feats[label_i].append(feat)

        for cls in classes:
            if len(by_class_images[cls]) == 0:
                continue
            feats = torch.stack(by_class_feats[cls])
            centroid = F.normalize(feats.mean(dim=0, keepdim=True), p=2, dim=1)
            scores = torch.sum(feats * centroid, dim=1)
            k = min(self.capacity_per_class, scores.numel())
            topk = torch.topk(scores, k=k, largest=True).indices.tolist()
            self.storage[cls] = [(by_class_images[cls][idx], float(scores[idx].item())) for idx in topk]

    def get_loader(
        self,
        batch_size: int = BATCH_SIZE,
        num_workers: int = NUM_WORKERS,
        shuffle: bool = True,
    ) -> Optional[DataLoader]:
        if not self.storage:
            return None

        images = []
        labels = []
        for cls in sorted(self.storage.keys()):
            for img, _ in self.storage[cls]:
                images.append(img)
                labels.append(cls)

        if not images:
            return None

        images_t = torch.stack(images)
        labels_t = torch.tensor(labels, dtype=torch.long)
        pin_memory = torch.cuda.is_available() and not FORCE_CPU
        return DataLoader(
            TensorDatasetWrapper(images_t, labels_t),
            batch_size=batch_size,
            shuffle=shuffle,
            num_workers=num_workers,
            pin_memory=pin_memory,
        )

    def __len__(self) -> int:
        return sum(len(v) for v in self.storage.values())


class IncrementalClassifier(nn.Module):
    def __init__(self, in_features: int, initial_classes: int = CLASSES_PER_TASK):
        super().__init__()
        self.in_features = in_features
        self.num_classes = initial_classes
        self.weight = nn.Parameter(torch.Tensor(self.num_classes, in_features))
        self.bias = nn.Parameter(torch.Tensor(self.num_classes))
        self.reset_parameters()

    def reset_parameters(self) -> None:
        nn.init.kaiming_uniform_(self.weight, a=np.sqrt(5))
        fan_in, _ = nn.init._calculate_fan_in_and_fan_out(self.weight)
        bound = 1 / np.sqrt(fan_in) if fan_in > 0 else 0
        nn.init.uniform_(self.bias, -bound, bound)

    def expand(self, n_new_classes: int) -> None:
        old_classes = self.num_classes
        new_total = old_classes + n_new_classes
        new_weight = nn.Parameter(torch.Tensor(new_total, self.in_features))
        new_bias = nn.Parameter(torch.Tensor(new_total))

        with torch.no_grad():
            new_weight[:old_classes].copy_(self.weight)
            new_bias[:old_classes].copy_(self.bias)
            nn.init.kaiming_uniform_(new_weight[old_classes:], a=np.sqrt(5))
            fan_in, _ = nn.init._calculate_fan_in_and_fan_out(new_weight[old_classes:])
            bound = 1 / np.sqrt(fan_in) if fan_in > 0 else 0
            nn.init.uniform_(new_bias[old_classes:], -bound, bound)

        self.num_classes = new_total
        self.weight = new_weight
        self.bias = new_bias

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return F.linear(x, self.weight, self.bias)


class ResNet18CIFAR(nn.Module):
    def __init__(self, initial_classes: int = CLASSES_PER_TASK):
        super().__init__()
        resnet = torchvision.models.resnet18(weights=None)
        resnet.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        resnet.maxpool = nn.Identity()
        self.features = nn.Sequential(*list(resnet.children())[:-1])
        self.classifier = IncrementalClassifier(512, initial_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        feat = self.features(x)
        feat = feat.flatten(1)
        logits = self.classifier(feat)
        return logits

    def expand_classifier(self, n_new_classes: int) -> None:
        self.classifier.expand(n_new_classes)

    def extract_features(self, x: torch.Tensor) -> torch.Tensor:
        feat = self.features(x)
        feat = feat.flatten(1)
        return feat


def evaluate(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
) -> float:
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            pred = logits.argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    return 100.0 * correct / total if total > 0 else 0.0


def train_task_1(
    model: nn.Module,
    train_loader: DataLoader,
    device: torch.device,
    amp_ok: bool,
) -> None:
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS_BASE)
    scaler = torch.amp.GradScaler("cuda") if amp_ok else None

    for epoch in range(EPOCHS_BASE):
        model.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            if amp_ok:
                with torch.amp.autocast("cuda"):
                    logits = model(x)
                    loss = F.cross_entropy(logits, y)
                scaler.scale(loss).backward()
                scaler.step(opt)
                scaler.update()
            else:
                logits = model(x)
                loss = F.cross_entropy(logits, y)
                loss.backward()
                opt.step()
        sched.step()


def train_incremental_task(
    model: nn.Module,
    train_loader: DataLoader,
    replay_loader: DataLoader,
    device: torch.device,
    amp_ok: bool,
) -> None:
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS_INCREMENTAL)
    scaler = torch.amp.GradScaler("cuda") if amp_ok else None

    for epoch in range(EPOCHS_INCREMENTAL):
        model.train()
        replay_iter = iter(replay_loader)
        for x_curr, y_curr in train_loader:
            x_curr, y_curr = x_curr.to(device), y_curr.to(device)

            try:
                x_replay, y_replay = next(replay_iter)
            except StopIteration:
                replay_iter = iter(replay_loader)
                x_replay, y_replay = next(replay_iter)
            x_replay, y_replay = x_replay.to(device), y_replay.to(device)

            x_combined = torch.cat([x_curr, x_replay], dim=0)
            y_combined = torch.cat([y_curr, y_replay], dim=0)

            opt.zero_grad()
            if amp_ok:
                with torch.amp.autocast("cuda"):
                    logits = model(x_combined)
                    loss = F.cross_entropy(logits, y_combined)
                scaler.scale(loss).backward()
                scaler.step(opt)
                scaler.update()
            else:
                logits = model(x_combined)
                loss = F.cross_entropy(logits, y_combined)
                loss.backward()
                opt.step()
        sched.step()


def run_baseline_replay(seed: int = 0, quick_run: bool = QUICK_RUN, save_path: str = SAVE_PATH) -> Dict:
    set_deterministic(seed)
    device = get_device(FORCE_CPU)
    amp_ok = USE_AMP and device.type == "cuda"

    print(f"🚀 Training Baseline Replay on Split CIFAR-100 | Seed: {seed} | Device: {device}")

    stats = ((0.5071, 0.4865, 0.4409), (0.2673, 0.2564, 0.2762))
    t_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.AutoAugment(transforms.AutoAugmentPolicy.CIFAR10),
        transforms.ToTensor(),
        transforms.Normalize(*stats),
    ])
    t_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(*stats),
    ])

    root = "./data"
    train_ds = torchvision.datasets.CIFAR100(root=root, train=True, download=True, transform=t_train)
    test_ds = torchvision.datasets.CIFAR100(root=root, train=False, download=True, transform=t_test)

    replay_buffer = DynamicReplayBuffer(REPLAY_BUFFER_CAPACITY)
    accuracy_matrix = np.zeros((N_TASKS, N_TASKS))
    results = {
        "accuracy_matrix": accuracy_matrix.tolist(),
        "average_accuracy": [],
        "tasks": [],
    }

    print("\n🔹 Task 1: Base Training (Classes 0-9)")
    train_loader, test_loader = get_task_loaders(train_ds, test_ds, task_id=1)
    model = ResNet18CIFAR(initial_classes=CLASSES_PER_TASK).to(device)
    train_task_1(model, train_loader, device, amp_ok)

    acc = evaluate(model, test_loader, device)
    accuracy_matrix[0, 0] = acc
    print(f"✅ Task 1 Accuracy: {acc:.2f}%")

    replay_buffer.add_exemplars_from_dataset(
        dataset=train_ds,
        classes=get_task_classes(1),
        feature_extractor=model,
        device=device,
    )

    results["average_accuracy"].append(float(acc))
    results["tasks"].append({
        "task_id": 1,
        "classes": get_task_classes(1),
    })
    results["accuracy_matrix"] = accuracy_matrix.tolist()
    save_progress(results, save_path)

    for task_id in range(2, N_TASKS + 1):
        class_start = CLASSES_PER_TASK * (task_id - 1)
        class_end = CLASSES_PER_TASK * task_id - 1
        print(f"\n🔹 Task {task_id}: Incremental Training (Classes {class_start}-{class_end})")
        current_classes = get_task_classes(task_id)
        train_loader, test_loader = get_task_loaders(train_ds, test_ds, task_id=task_id)

        print("  🔸 Expanding classifier head")
        model.expand_classifier(CLASSES_PER_TASK)
        model = model.to(device)

        replay_loader = replay_buffer.get_loader()
        if replay_loader is None:
            raise RuntimeError("Replay buffer is empty before incremental training.")

        print("  🔸 Training on combined current + replay data")
        train_incremental_task(model, train_loader, replay_loader, device, amp_ok)

        replay_buffer.add_exemplars_from_dataset(
            dataset=train_ds,
            classes=current_classes,
            feature_extractor=model,
            device=device,
        )

        print("  🔸 Evaluation")
        for t in range(1, task_id + 1):
            _, test_loader_t = get_task_loaders(train_ds, test_ds, task_id=t)
            task_acc = evaluate(model, test_loader_t, device)
            accuracy_matrix[task_id - 1, t - 1] = task_acc
            print(f"    Task {t} Accuracy: {task_acc:.2f}%")

        avg_acc = float(np.mean(accuracy_matrix[task_id - 1, :task_id]))
        results["average_accuracy"].append(avg_acc)
        results["accuracy_matrix"] = accuracy_matrix.tolist()
        results["tasks"].append({
            "task_id": task_id,
            "classes": current_classes,
        })
        save_progress(results, save_path)

    print("\n🎉 Training Complete!")
    print(f"📊 Final Accuracy Matrix:\n{np.array(results['accuracy_matrix'])}")
    print(f"📈 Average Accuracy: {results['average_accuracy'][-1]:.2f}%")
    return results


if __name__ == "__main__":
    run_baseline_replay(seed=0, quick_run=QUICK_RUN)


In [ ]:
'''
✅ Task 1 Accuracy: 92.50%

🔹 Task 2: Incremental Training (Classes 10-19)
  🔸 Expanding classifier head
  🔸 Training on combined current + replay data
  🔸 Evaluation
    Task 1 Accuracy: 2.90%
    Task 2 Accuracy: 90.30%

🔹 Task 3: Incremental Training (Classes 20-29)
  🔸 Expanding classifier head
  🔸 Training on combined current + replay data
  🔸 Evaluation
    Task 1 Accuracy: 5.30%
    Task 2 Accuracy: 1.70%
    Task 3 Accuracy: 90.60%

🔹 Task 4: Incremental Training (Classes 30-39)
  🔸 Expanding classifier head
  🔸 Training on combined current + replay data
  🔸 Evaluation
    Task 1 Accuracy: 4.90%
    Task 2 Accuracy: 0.20%
    Task 3 Accuracy: 5.30%
    Task 4 Accuracy: 92.50%

🔹 Task 5: Incremental Training (Classes 40-49)
  🔸 Expanding classifier head
  🔸 Training on combined current + replay data
  🔸 Evaluation
    Task 1 Accuracy: 4.50%
    Task 2 Accuracy: 0.60%
    Task 3 Accuracy: 1.80%
    Task 4 Accuracy: 4.70%
    Task 5 Accuracy: 91.90%

🔹 Task 6: Incremental Training (Classes 50-59)
  🔸 Expanding classifier head
  🔸 Training on combined current + replay data
  🔸 Evaluation
    Task 1 Accuracy: 1.70%
    Task 2 Accuracy: 0.40%
    Task 3 Accuracy: 3.40%
    Task 4 Accuracy: 2.20%
    Task 5 Accuracy: 9.80%
    Task 6 Accuracy: 90.20%

🔹 Task 7: Incremental Training (Classes 60-69)
  🔸 Expanding classifier head
  🔸 Training on combined current + replay data
  🔸 Evaluation
    Task 1 Accuracy: 2.60%
    Task 2 Accuracy: 0.10%
    Task 3 Accuracy: 1.90%
    Task 4 Accuracy: 1.30%
    Task 5 Accuracy: 5.90%
    Task 6 Accuracy: 16.60%
    Task 7 Accuracy: 91.50%
🔹 Task 8: Incremental Training (Classes 70-79)
  🔸 Expanding classifier head
  🔸 Training on combined current + replay data
  🔸 Evaluation
    Task 1 Accuracy: 1.90%
    Task 2 Accuracy: 0.10%
    Task 3 Accuracy: 0.20%
    Task 4 Accuracy: 0.80%
    Task 5 Accuracy: 3.50%
    Task 6 Accuracy: 12.40%
    Task 7 Accuracy: 4.80%
    Task 8 Accuracy: 90.70%

🔹 Task 9: Incremental Training (Classes 80-89)
  🔸 Expanding classifier head
  🔸 Training on combined current + replay data
  🔸 Evaluation
    Task 1 Accuracy: 1.10%
    Task 2 Accuracy: 0.20%
    Task 3 Accuracy: 1.40%
    Task 4 Accuracy: 0.30%
    Task 5 Accuracy: 3.70%
    Task 6 Accuracy: 5.00%
    Task 7 Accuracy: 7.70%
    Task 8 Accuracy: 10.20%
    Task 9 Accuracy: 93.80%

🔹 Task 10: Incremental Training (Classes 90-99)
  🔸 Expanding classifier head
  🔸 Training on combined current + replay data
  🔸 Evaluation
    Task 1 Accuracy: 2.10%
    Task 2 Accuracy: 0.10%
    Task 3 Accuracy: 2.40%
    Task 4 Accuracy: 0.30%
    Task 5 Accuracy: 3.40%
    Task 6 Accuracy: 4.90%
    Task 7 Accuracy: 3.50%
    Task 8 Accuracy: 5.30%
    Task 9 Accuracy: 3.10%
    Task 10 Accuracy: 94.90%

🎉 Training Complete!
📊 Final Accuracy Matrix:
[[92.5  0.   0.   0.   0.   0.   0.   0.   0.   0. ]
 [ 2.9 90.3  0.   0.   0.   0.   0.   0.   0.   0. ]
 [ 5.3  1.7 90.6  0.   0.   0.   0.   0.   0.   0. ]
 [ 4.9  0.2  5.3 92.5  0.   0.   0.   0.   0.   0. ]
 [ 4.5  0.6  1.8  4.7 91.9  0.   0.   0.   0.   0. ]
 [ 1.7  0.4  3.4  2.2  9.8 90.2  0.   0.   0.   0. ]
 [ 2.6  0.1  1.9  1.3  5.9 16.6 91.5  0.   0.   0. ]
 [ 1.9  0.1  0.2  0.8  3.5 12.4  4.8 90.7  0.   0. ]
 [ 1.1  0.2  1.4  0.3  3.7  5.   7.7 10.2 93.8  0. ]
 [ 2.1  0.1  2.4  0.3  3.4  4.9  3.5  5.3  3.1 94.9]]
📈 Average Accuracy: 12.00%

'''